In [3]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os

print("\n" + "="*70)
print("DATA MERGING - CLEAN LABELS + ALL FEATURES")
print("="*70)

# ============================================================
# Step 1: 加载clean labels
# ============================================================

print("\nStep 1: Loading clean Y labels...")

try:
    df_labels = pd.read_parquet('amc_final_with_EWS_labels_TIME_LAGGED.parquet')
    df_labels['timestamp'] = pd.to_datetime(df_labels['timestamp'])
    
    if df_labels['timestamp'].dt.tz is not None:
        df_labels['timestamp'] = df_labels['timestamp'].dt.tz_localize(None)
    
    print(f"  ✓ Loaded: {df_labels.shape[0]:,} rows × {df_labels.shape[1]} columns")
    print(f"  Date range: {df_labels['timestamp'].min()} to {df_labels['timestamp'].max()}")
    
    print(f"\n  EWS Labels distribution:")
    for label in ['EWS_5min', 'EWS_15min', 'EWS_30min']:
        if label in df_labels.columns:
            count = df_labels[label].sum()
            pct = count / len(df_labels) * 100
            print(f"    {label}: {int(count):>6,} ({pct:>5.2f}%)")
        else:
            print(f"    {label}: NOT FOUND")
    
except FileNotFoundError:
    print("\n  ✗ ERROR: amc_final_with_EWS_labels_TIME_LAGGED.parquet not found!")
    print("  Please run clean_labels_amc.py first")
    exit(1)

# ============================================================
# Step 2: 加载所有feature files
# ============================================================

print("\n" + "="*70)
print("Step 2: Loading all feature files...")
print("="*70)

feature_files = {
    'cascade':          'cascade_features_amc_5min.parquet',
    'network':          'network_features_amc_5min.parquet',
    'temporal':         'temporal_features_amc_5min.parquet',
    'text':             'text_features_amc_5min.parquet',
    'burstiness':       'burstiness_features_amc_5min.parquet',
    'user_overlap':     'user_overlap_features_amc_5min.parquet',
    'text_duplication': 'text_duplication_features_amc_5min.parquet',
}

feature_data = {}
missing_files = []

for name, filepath in feature_files.items():
    if os.path.exists(filepath):
        df = pd.read_parquet(filepath)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        if df['timestamp'].dt.tz is not None:
            df['timestamp'] = df['timestamp'].dt.tz_localize(None)
        feature_data[name] = df
        print(f"  ✓ {name:20} {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols - {filepath}")
    else:
        print(f"  ✗ {name:20} MISSING - {filepath}")
        missing_files.append(filepath)

if missing_files:
    print(f"\n  ⚠️  WARNING: {len(missing_files)} feature files missing!")
    for f in missing_files:
        print(f"    - {f}")
    print("\n  自动继续（调度模式）...")

print(f"\n  ✓ Loaded {len(feature_data)} feature files")

# ============================================================
# Step 3: 合并所有features
# ============================================================

print("\n" + "="*70)
print("Step 3: Merging all features with clean labels...")
print("="*70)

merged = df_labels.copy()
print(f"\n  Starting with: {merged.shape[1]} columns (labels + market)")

for name, df_feat in feature_data.items():
    print(f"\n  Merging {name}...")
    before_cols = len(merged.columns)
    feat_cols = [c for c in df_feat.columns if c != 'timestamp']
    merged = merged.merge(
        df_feat[['timestamp'] + feat_cols],
        on='timestamp', how='left', suffixes=('', f'_{name}')
    )
    after_cols = len(merged.columns)
    print(f"    + {after_cols - before_cols:>2} new features → total: {after_cols}")

print(f"\n  Final shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns")

# ============================================================
# Step 4: 识别feature groups（显式列表，无关键词匹配）
# ============================================================

print("\n" + "="*70)
print("Step 4: Categorizing features into groups...")
print("="*70)

exclude_cols = ['timestamp', 'EWS_5min', 'EWS_15min', 'EWS_30min']

feature_groups = {
    'market': [], 'cascade': [], 'network': [], 'temporal': [],
    'text': [], 'burstiness': [], 'user_overlap': [], 'text_duplication': []
}

NETWORK_FEATURES = [
    'node_count', 'edge_count', 'num_components', 'largest_component_size',
    'density', 'avg_degree', 'max_degree', 'avg_betweenness', 'max_betweenness',
    'avg_clustering', 'max_k_core', 'avg_pagerank', 'max_pagerank',
]
TEMPORAL_FEATURES = [
    'unique_users', 'total_score', 'avg_score', 'max_score',
    'user_growth', 'is_burst', 'burst_intensity', 'pct_posts_under_5sec',
    'volume', 'volume_mean', 'volume_std', 'volume_z', 'is_volume_surge',
    'volume_lag5min', 'volume_z_lag5min',
    'post_volume', 'comment_volume', 'total_volume',
    'post_velocity', 'comment_velocity', 'total_velocity',
    'post_acceleration', 'comment_acceleration', 'total_acceleration',
    'volume_mean_60min', 'volume_std_60min', 'volume_zscore',
    'post_volume_5min', 'comment_volume_5min', 'total_volume_5min',
    'post_volume_15min', 'comment_volume_15min', 'total_volume_15min',
    'post_volume_30min', 'comment_volume_30min', 'total_volume_30min',
]
TEXT_FEATURES = [
    'emoji_density', 'caps_ratio', 'exclamation_ratio',
    'sentiment_positive', 'sentiment_negative', 'sentiment_neutral',
    'sentiment_compound', 'urgency_score', 'bullish_keywords', 'bearish_keywords',
]
CASCADE_FEATURES = [
    'cascade_size_mean', 'cascade_size_max', 'cascade_size_sum',
    'cascade_depth_mean', 'cascade_depth_max',
    'cascade_breadth_mean', 'cascade_breadth_max',
    'structural_virality_mean', 'structural_virality_max',
    'adoption_speed_mean', 'adoption_speed_max', 'cascade_count',
]
BURSTINESS_FEATURES = [
    'burstiness_coefficient', 'sync_posting_rate',
    'inter_arrival_mean', 'inter_arrival_std', 'inter_arrival_min',
]
USER_OVERLAP_FEATURES = [
    'cross_thread_overlap', 'multi_thread_user_ratio',
    'coordinated_group_size', 'unique_threads', 'thread_concentration',
]
TEXT_DUPLICATION_FEATURES = [
    'exact_duplicate_rate', 'fuzzy_duplicate_rate', 'unique_text_ratio',
    'avg_text_similarity', 'max_duplicate_count', 'duplicate_cluster_size',
]

for col in merged.columns:
    if col in exclude_cols:
        continue
    if col in NETWORK_FEATURES:
        feature_groups['network'].append(col)
    elif col in TEMPORAL_FEATURES:
        feature_groups['temporal'].append(col)
    elif col in TEXT_FEATURES:
        feature_groups['text'].append(col)
    elif col in CASCADE_FEATURES:
        feature_groups['cascade'].append(col)
    elif col in BURSTINESS_FEATURES:
        feature_groups['burstiness'].append(col)
    elif col in USER_OVERLAP_FEATURES:
        feature_groups['user_overlap'].append(col)
    elif col in TEXT_DUPLICATION_FEATURES:
        feature_groups['text_duplication'].append(col)
    else:
        feature_groups['market'].append(col)

print("\nFeature group summary:")
total_features = 0
for group, features in feature_groups.items():
    print(f"  {group:20} {len(features):>3} features")
    total_features += len(features)
print(f"\n  Total features: {total_features}")

# ============================================================
# Step 5: 标准化
# ============================================================

print("\n" + "="*70)
print("Step 5: Standardizing features...")
print("="*70)

label_cols = ['timestamp', 'EWS_5min', 'EWS_15min', 'EWS_30min']
feature_cols = [c for c in merged.columns if c not in label_cols]

X_features = merged[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)
merged_scaled = pd.DataFrame(X_scaled, columns=feature_cols, index=merged.index)
for col in label_cols:
    if col in merged.columns:
        merged_scaled[col] = merged[col].values

print(f"  ✓ Standardization complete")

# ============================================================
# Step 6: 保存
# ============================================================

print("\n" + "="*70)
print("Step 6: Saving outputs...")
print("="*70)

os.makedirs('data', exist_ok=True)

merged.to_parquet('data/merged_data_amc_raw_CLEAN.parquet', index=False)
print(f"  ✓ merged_data_amc_raw_CLEAN.parquet: {merged.shape[0]:,} rows × {merged.shape[1]} cols")

merged_scaled.to_parquet('data/merged_data_amc_scaled_CLEAN.parquet', index=False)
print(f"  ✓ merged_data_amc_scaled_CLEAN.parquet: {merged_scaled.shape[0]:,} rows × {merged_scaled.shape[1]} cols")

with open('data/feature_groups_amc_CLEAN.json', 'w', encoding='utf-8') as f:
    json.dump(feature_groups, f, indent=2)
print(f"  ✓ feature_groups_amc_CLEAN.json: {len(feature_groups)} groups")

print("\n" + "="*70)
print("DATA MERGING COMPLETE — AMC")
print("="*70)
print(f"\n  Total rows: {merged.shape[0]:,}")
print(f"  Total features: {total_features}")
print(f"  Output: data/merged_data_amc_scaled_CLEAN.parquet")


DATA MERGING - CLEAN LABELS + ALL FEATURES

Step 1: Loading clean Y labels...
  ✓ Loaded: 67,510 rows × 33 columns
  Date range: 2019-07-01 12:30:00 to 2021-07-01 00:00:00

  EWS Labels distribution:
    EWS_5min:  1,498 ( 2.22%)
    EWS_15min:  2,820 ( 4.18%)
    EWS_30min:  4,746 ( 7.03%)

Step 2: Loading all feature files...
  ✓ cascade               4,464 rows ×  13 cols - cascade_features_amc_5min.parquet
  ✓ network              46,235 rows ×  14 cols - network_features_amc_5min.parquet
  ✓ temporal             46,235 rows ×  29 cols - temporal_features_amc_5min.parquet
  ✓ text                 46,235 rows ×  11 cols - text_features_amc_5min.parquet
  ✓ burstiness           46,235 rows ×   7 cols - burstiness_features_amc_5min.parquet
  ✓ user_overlap         46,235 rows ×   6 cols - user_overlap_features_amc_5min.parquet
  ✓ text_duplication     46,235 rows ×   7 cols - text_duplication_features_amc_5min.parquet

  ✓ Loaded 7 feature files

Step 3: Merging all features with cle